In [ ]:
import os

class MiniLSMTree:
    def __init__(self, max_memtable_size=5):
        """
        Инициализация структуры LSM-дерева.
        max_memtable_size - искусственно заниженный лимит ключей 
        для демонстрации работы flush() (в реальности это мегабайты).
        """
        self.max_memtable_size = max_memtable_size
        
        # MemTable: структура данных в оперативной памяти (для поиска O(1) используем словарь)
        self.memtable = {}
        
        # Список файлов SSTable на диске (от старых к новым)
        self.sstable_files = []
        
        # Директория для хранения файлов
        self.data_dir = "lsm_data"
        if not os.path.exists(self.data_dir):
            os.makedirs(self.data_dir)
            
        print("LSM-Tree DB инициализирована.")

    def put(self, key, value):
        """Запись данных. Происходит мгновенно, так как пишем только в ОЗУ."""
        # ========================================================
        # TODO 2.1: Добавьте или обновите ключ в само_м словаре self.memtable
        # ========================================================
        pass 
        
        print(f"PUT: {key} -> {value} (в MemTable)")
        
        # ========================================================
        # TODO 2.2: Проверьте, если размер self.memtable >= self.max_memtable_size,
        # вызовите метод self.flush()
        # ========================================================
        pass

    def flush(self):
        """Сброс оперативной памяти на диск (создание SSTable)."""
        if not self.memtable:
            return

        sstable_idx = len(self.sstable_files)
        filepath = os.path.join(self.data_dir, f"sstable_{sstable_idx}.txt")
        
        # ========================================================
        # TODO 2.3: SSTable (Sorted String Table) ОБЯЗАНА быть отсортирована по ключу.
        # Получите список ключей из self.memtable и отсортируйте его.
        # ========================================================
        sorted_keys = [] # Замените на вашу логику сортировки
        
        # Записываем отсортированные данные на диск
        with open(filepath, 'w') as f:
            for k in sorted_keys:
                f.write(f"{k},{self.memtable[k]}\n")
                
        self.sstable_files.append(filepath)
        print(f"\n[FLUSH] MemTable переполнен. Данные сброшены в диск: {filepath}")
        
        # ========================================================
        # TODO 2.4: Очистите self.memtable (подготовьте к новым данным)
        # ========================================================
        pass

    def get(self, key):
        """Поиск ключа. Это самое узкое (медленное) место LSM-деревьев."""
        print(f"\nGET: Поиск ключа '{key}'...")
        
        # ========================================================
        # TODO 2.5: Шаг 1. Ищем горячие данные в оперативной памяти.
        # Проверьте, есть ли ключ в self.memtable. Если есть - верните его значение.
        # ========================================================
        pass
            
        # ========================================================
        # Шаг 2. Если в ОЗУ нет, идем искать на диск (в SSTables).
        # TODO 2.6: ВНИМАНИЕ! Ключевая концепция: старые файлы содержат устаревшие данные.
        # Чтобы найти самую свежую версию ключа, мы ДОЛЖНЫ обходить список 
        # self.sstable_files в ОБРАТНОМ порядке (от новых к старым).
        # Напишите цикл, проходящий по файлам в обратном порядке:
        # ========================================================
        
        # for filepath in ... : # Ваш цикл здесь
        #     with open(filepath, 'r') as f:
        #         for line in f:
        #             k, v = line.strip().split(',')
        #             if k == str(key):
        #                 print(f"Найдено на диске (история): {filepath}")
        #                 return v
                        
        print("Ключ не найден.")
        return None


if __name__ == "__main__":
    db = MiniLSMTree(max_memtable_size=3)
    
    # Первая пачка данных
    db.put("user1", "Alex_v1")
    db.put("user2", "Bob")
    db.put("user3", "Charlie") # Здесь должен сработать Flush (т.к. лимит 3)
    
    # Вторая пачка данных (обновляем user1)
    db.put("user1", "Alex_v2") 
    db.put("user4", "Dave")
    db.put("user5", "Eve")     # И снова Flush
    
    # Третья пачка данных (в MemTable, без флаша)
    db.put("user1", "Alex_FINAL_VERSION")
    db.put("user6", "Frank")
    
    # Тестируем чтение
    result = db.get("user1")
    print(f"Результат get('user1'): {result} (Ожидается: Alex_FINAL_VERSION)")
    
    result = db.get("user4")
    print(f"Результат get('user4'): {result} (Ожидается: Dave)")
